# Computing $\varepsilon_{\rm coh}$ for the $N=20$ mirror active-pair code


**Goal.** Independently reproduce the coherent (noise-free) silent error $\varepsilon_{\rm coh}$ quoted in the paper ($\simeq 2\times10^{-7}$, Eq. S61), together with the end-of-round leakage $\bar\ell(\tau)$ ($3.3\times10^{-3}$ without echo, $\approx 7\times10^{-3}$ with echo).

**Why check it.** The SI bounds $\varepsilon_{\rm coh}=\mathcal O(\ell_{\max}^2)+4g_d^2/\delta^2$ (processes A3 and A4).

- A4 is tiny: ED gives a transfer of $\sim5\times10^{-15}$.
- A3 is set by the leakage. With $\ell\sim3$–$7\times10^{-3}$, $\ell^2\sim10^{-5}$, which is the same size as the headline $\varepsilon\simeq2.6\times10^{-5}$.
- In particular, the A3 phase term is $\theta^2/4$ with $\theta\sim\ell_{\max}$.

So $2\times10^{-7}$ needs the phase and modulus differences between the two codewords to be much smaller than $\ell$. That is plausible, but it should be confirmed with the same protocol (logical echo) that the headline uses.

**Parameters** (as in the paper): $N=20$, $\alpha=0.4$, $J_z/J_x=100$, $J_x\tau=10$. The codewords are
$S=\{1,2,16,18\}$ and $S'=\{1,5,18,19\}$, with $|0_L\rangle=|v_S\rangle$ and $|1_L\rangle=|v_{S'}\rangle$.

## 1. What is simulated, and why it is exact for this quantity

$\varepsilon_{\rm coh}$ is the error with **no kicks**: only coherent dynamics, checks at the end, and the logical echo.

**In-sector Hamiltonian.** We use $H_{\rm eff}^{b}=T+U$, i.e. $H_0$ plus the number-conserving hopping $J_x(a_i^\dagger a_{i+1}+{\rm h.c.})$. The sector-changing terms ($B_x$, pair creation) give flagged leakage $P^{\rm sector}_{\rm leak}$, which is counted separately in $\bar\ell$ and in A2/A5. This matches "exact diagonalization of the in-sector dynamics" in the SI.

**Echo.** At $\tau/2$ and at $\tau$ we apply $F_D=\prod_{p\in D}X_{p_L}X_{p_R}$, where $D=S\triangle S'$ as pairs. Here that is the pairs $(2,19)$ and $(5,16)$, so the pulse flips sites $\{2,5,16,19\}$.

**Why restricting to $b=4$ is exact.** On a leaked component (e.g. a pair holding 0 or 2 excitations), $F_D$ can change $b$. Such a component then evolves in another sector, because $T+U$ conserves $b$. The second pulse $F_D$ maps codespace to codespace and is an involution. A component can therefore end in the codespace only if it was in the codespace just before the second pulse, and that requires $b=4$. Dropping amplitude that the first pulse sends out of $b=4$ thus changes nothing in the accepted part. That accepted part, $P\,V\,P$, is all that $\varepsilon$ depends on.

**Dimension.** $\binom{20}{4}=4845$, so full dense diagonalization is cheap.

In [1]:
import numpy as np, itertools

N, alpha, Jz, Jx = 20, 0.4, 100.0, 1.0   # units of J_x
tau = 10.0                                # J_x * tau
S  = (1, 2, 16, 18)                       # first branch of |0_L>
Sp = (1, 5, 18, 19)                       # first branch of |1_L>
b  = 4

mirror = lambda c: tuple(sorted(N + 1 - i for i in c))

# Basis of the b=4 sector: sorted tuples of occupied (down-spin) sites, 1-indexed
cfgs = list(itertools.combinations(range(1, N + 1), b))
idx  = {c: k for k, c in enumerate(cfgs)}
D = len(cfgs)
print("sector dimension:", D)

sector dimension: 4845


## 2. Hamiltonian

**Diagonal part.** We use $E(c)=\sum_{i<j}J_z|i-j|^{-\alpha}\sigma^z_i\sigma^z_j$, with $\sigma^z=-1$ on occupied sites. This is the exact Ising energy; it equals $\langle c|U|c\rangle+E_{\rm vac}$ in the hard-core language.

**Off-diagonal part.** $J_x$ couples configurations that differ by moving one excitation to an empty neighbouring site. In the Holstein–Primakoff form this is $J_x(a_i^\dagger a_{i+1}+{\rm h.c.})$, with amplitude $+J_x$ and no fermionic signs (hard-core bosons).

In [2]:
# Ising coupling matrix, vectorised
r = np.arange(1, N + 1)
Jmat = Jz / np.abs(r[:, None] - r[None, :]).astype(float) ** alpha
np.fill_diagonal(Jmat, 0.0)

def energy(c):
    s = np.ones(N); s[np.array(c) - 1] = -1.0
    return 0.5 * s @ Jmat @ s            # = sum_{i<j} J_ij s_i s_j

H = np.zeros((D, D))
for k, c in enumerate(cfgs):
    H[k, k] = energy(c)
    occ = set(c)
    for i in c:
        for j in (i - 1, i + 1):
            if 1 <= j <= N and j not in occ:
                new = tuple(sorted((occ - {i}) | {j}))
                H[k, idx[new]] += Jx

assert np.allclose(H, H.T)
w, V = np.linalg.eigh(H)                  # exact spectrum of T+U in the b=4 sector

print("delta = E(S') - E(S) =", energy(Sp) - energy(S), " (paper: -71.5 J_x)")

/var/folders/hx/v2z4z5_d7zzcmvpt7g3tzrs40000gn/T/ipykernel_17988/3047745852.py:3: RuntimeWarning: divide by zero encountered in divide
  Jmat = Jz / np.abs(r[:, None] - r[None, :]).astype(float) ** alpha


delta = E(S') - E(S) = -71.49955863355581  (paper: -71.5 J_x)


## 3. Codewords, codespace projector and echo pulse

- **Codewords.** $|v_S\rangle=(|S\rangle+|\mathcal R S\rangle)/\sqrt2$, where $\mathcal R S$ is the mirror configuration.
- **Codespace projector.** The checks (pair totals, $\hat s_1\hat s_4=-1$, $\hat s_2\hat s_3=-1$, $\mathcal R=+1$) select exactly $\mathrm{span}\{|v_S\rangle,|v_{S'}\rangle\}$. So $P=\sum_{L}|v_L\rangle\langle v_L|$, and we store it as the $2\times D$ matrix `Pm` of logical rows.
- **Echo pulse $F_D$.** This is a permutation on configurations. A configuration is sent to the symmetric difference with $\{2,5,16,19\}$. If the result has $b\neq4$ it leaves the sector and, by the argument in Section 1, is dropped.

In [3]:
def codeword(c):
    v = np.zeros(D)
    v[idx[c]] += 1 / np.sqrt(2)
    v[idx[mirror(c)]] += 1 / np.sqrt(2)
    return v

logical = [codeword(S), codeword(Sp)]
Pm = np.array(logical)                    # rows <0_L|, <1_L|
assert np.allclose(Pm @ Pm.T, np.eye(2))

# Echo pulse: flip both sites of every active pair in D = S (triangle) S'
pulse_sites = {2, 19, 5, 16}
assert tuple(sorted(set(S) ^ pulse_sites)) == Sp   # F_D maps S -> S'

perm = np.full(D, -1)
for k, c in enumerate(cfgs):
    new = tuple(sorted(set(c) ^ pulse_sites))
    if len(new) == b:
        perm[k] = idx[new]

def F(psi):
    out = np.zeros(D, complex)
    keep = perm >= 0
    out[perm[keep]] = psi[keep]
    return out

def evolve(t, psi):
    return V @ (np.exp(-1j * w * t) * (V.T @ psi))

## 4. The accepted operator and the error formula

This follows SI Step 2. For a history with evolution $V_h$, the accepted operator is $\mathcal A=U_L^\dagger P V_h P$, a $2\times2$ matrix in the logical basis. Its column $L$ is $P\,V_h|L\rangle$. With a maximally entangled reference (SI Eq. S36):
$$
p_{\rm acc}=\tfrac12\mathrm{Tr}\,\mathcal A^\dagger\mathcal A,\qquad
\varepsilon=\tfrac12\mathrm{Tr}\,\mathcal A^\dagger\mathcal A-\tfrac14|\mathrm{Tr}\,\mathcal A|^2 .
$$
This is invariant under a global phase on $\mathcal A$.

**With the echo.** Ideally $F_D\,e^{-iH_L\tau/2}F_D\,e^{-iH_L\tau/2}=e^{-i(E_0+E_1)\tau/2}\,I$. So $U_L$ is a global phase and we can use $\mathcal A=P\,F_D U(\tau/2) F_D U(\tau/2)\,P$ directly. Any phase that is constant over the round cancels without knowing it.

**Without the echo**, for comparison, we must track the logical frequency (SI Remark 1). We take $U_L=\mathrm{diag}(e^{-iE_0\tau},e^{-iE_1\tau})$, where $E_L$ is the exact eigenvalue of the dressed eigenstate with the largest overlap on $|v_L\rangle$. That is ideal tracking: it includes all hopping shifts.

**Leakage.** $\bar\ell(\tau)=1-p_{\rm acc}$ for the noiseless round, i.e. the mean of $\ell_S$ and $\ell_{S'}$.

In [4]:
def metrics(A):
    tr2 = np.trace(A.conj().T @ A).real
    return {"eps_coh": 0.5 * tr2 - 0.25 * abs(np.trace(A)) ** 2,
            "leak":    1 - 0.5 * tr2}

def A_echo(t):
    A = np.zeros((2, 2), complex)
    for L, v in enumerate(logical):
        psi = F(evolve(t / 2, F(evolve(t / 2, v.astype(complex)))))
        A[:, L] = Pm @ psi
    return A

E_dressed = np.array([w[np.argmax(np.abs(V.T @ v))] for v in logical])

def A_noecho(t):
    A = np.zeros((2, 2), complex)
    for L, v in enumerate(logical):
        A[:, L] = Pm @ evolve(t, v.astype(complex))
    return np.diag(np.exp(1j * E_dressed * t)) @ A     # remove tracked U_L

## 5. Results at $J_x\tau=10$, and robustness in $\tau$

$\varepsilon_{\rm coh}(\tau)$ oscillates at the hop gaps ($\sim J_z$), so a single $\tau$ can land on a lucky or unlucky point. We therefore report:

- the value at $\tau=10$;
- the mean and maximum over a window $J_x\tau\in[9,11]$, which is many oscillation periods. The maximum is the honest worst case to quote.

**Sanity checks** against the paper: leakage $\approx3.3\times10^{-3}$ without the echo and $\approx7\times10^{-3}$ with it, and $\delta=-71.5\,J_x$.

In [ ]:
ts = np.linspace(9, 11, 801)
for name, Af in [("echo", A_echo), ("no echo (ideal tracking)", A_noecho)]:
    m0 = metrics(Af(tau))
    scan = np.array([[metrics(Af(t))[k] for k in ("eps_coh", "leak")] for t in ts])
    print(f"{name}")
    print(f"  tau=10      : eps_coh = {m0['eps_coh']:.2e}   leak = {m0['leak']:.2e}")
    print(f"  tau in[9,11]: eps_coh mean = {scan[:,0].mean():.2e}, max = {scan[:,0].max():.2e}"
          f";  leak mean = {scan[:,1].mean():.2e}")

## 6. Decompose the error: tilt, phase, bit flip

Write $\mathcal A=\begin{pmatrix}a&u\\v&b\end{pmatrix}$ (SI Eq. S40). Then $\varepsilon=\tfrac14|a-b|^2+\tfrac12(|u|^2+|v|^2)$. We split this into three parts:

- **modulus (tilt):** $\tfrac14(|a|-|b|)^2\approx\tfrac1{16}(\ell_S-\ell_{S'})^2$;
- **phase:** $\approx\tfrac14|a||b|\,\theta^2$, the residual logical $Z$;
- **bit flip:** $\tfrac12(|u|^2+|v|^2)$, i.e. A4.

This shows which term sets $\varepsilon_{\rm coh}$. It also tests whether the paper's $\mathcal O(\ell_{\max}^2)$ estimate is saturated or strongly beaten.

In [ ]:
def decompose(A):
    a, u, v, bb = A[0, 0], A[0, 1], A[1, 0], A[1, 1]
    theta = np.angle(a * np.conj(bb))
    return {"modulus": 0.25 * (abs(a) - abs(bb)) ** 2,
            "phase":   0.25 * abs(a) * abs(bb) * 2 * (1 - np.cos(theta)),  # exact: |a-b|^2 split
            "bitflip": 0.5 * (abs(u) ** 2 + abs(v) ** 2),
            "ell_S": 1 - abs(a) ** 2 - abs(v) ** 2,
            "ell_S'": 1 - abs(bb) ** 2 - abs(u) ** 2}

for name, Af in [("echo", A_echo), ("no echo", A_noecho)]:
    print(name, {k: f"{x:.2e}" for k, x in decompose(Af(tau)).items()})

## 7. Optional: include the sector-changing terms

The sections above reproduce the SI's "in-sector" definition. To check that $B_x$ and pair creation don't add a silent part, repeat the calculation with the full $H$ (Eq. 1) at some $B_x/J_x$ (the paper doesn't fix it for the memory section; e.g. 0.1). The full $2^{20}$ space is too large for dense diagonalization. Instead, use the sectors $b=3,4,5$ (dimension $\approx 3\times10^4$), which capture first-order leakage, and evolve with `scipy.sparse.linalg.expm_multiply`.

Note: at the pulse the flips can move amplitude between sectors, so keep all three sectors through the round. Expect the extra contribution to be $\mathcal O\big((P^{\rm sector}_{\rm leak})^2\big)$.

## 8. How to read the outcome

- **If the echoed $\varepsilon_{\rm coh}$, at its scan maximum, is $\lesssim10^{-6}$:** the headline $\varepsilon\simeq2.6\times10^{-5}$ stands. Quote the window maximum rather than a single $\tau$.
- **If it comes out at $\sim10^{-5}$:** it becomes comparable to the two stochastic terms. Then:
  - update Eq. S61 and the main-text $\varepsilon$ (e.g. $\varepsilon\approx3.5$–$4\times10^{-5}$);
  - revise the ratio $\varepsilon_B/\varepsilon$ (it would fall from $\sim76$ toward $\sim50$, so "more than 50-fold" would no longer be safe);
  - recompute the conditional infidelity $\varepsilon/p_{\rm acc}$.
- **Either way,** state in the SI which protocol (echo or not) and which $\tau$ (or window) $\varepsilon_{\rm coh}$ refers to. The current text doesn't say.